# Politician Polarization Analysis

Standalone, Google Colab-compatible analysis of partisan structure in LLM activations.

**Pipeline:** For each GSS topic, generate prompts for ~550 Congress members (116th Congress), extract attention-head activations at the last token, compute per-head metrics, correlate topic-level activation polarization with GSS survey polarization.

**Metrics per (layer, head):**
- `Mahalanobis` — D/R centroid distance in PCA-15 space, pooled covariance
- `Total_Dispersion` — Σλᵢ (sum of eigenvalues; overall activation spread)
- `Intrinsic_Dim` — (Σλᵢ)²/Σλᵢ² (participation ratio; effective dimensionality)
- `PC1_Ratio` — λ₁/Σλᵢ (fraction of variance in first PC)
- `Davies_Bouldin` — DB cluster separation score (lower = better D/R separation)

**Default model:** `cognitivecomputations/dolphin-2.9.1-yi-1.5-34b` (Yi-1.5-34B, Dolphin FT)

In [ ]:
# ── Install dependencies ──────────────────────────────────────────────────────
import subprocess, sys
def pip(*pkgs):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *pkgs])
pip('torch', 'transformers>=4.40', 'accelerate', 'bitsandbytes',
    'scikit-learn', 'pandas', 'numpy', 'plotly', 'scipy', 'joblib')
print('Dependencies ready.')

In [ ]:
# ── Configuration ─────────────────────────────────────────────────────────────
CONDITIONS   = ['rhetorical', 'stance', 'survey']   # all three will be run per model
CATEGORIES   = ['public', 'private']

PCA_DIM       = 15   # PCA dims used for Mahalanobis distance
PCA_SAVE      = 30   # PCA dims saved per head for future recomputation
EIGEN_PCA_DIM = 10
MAX_LENGTH    = 128

SYSTEM_MSG = 'You are simulating the public stance of U.S. politicians.\n\n'

LOCAL_POLITICIAN_CSV = '/project/jevans/maxzhuyt/gss_polarization/data/politicians.csv'
VOTEVIEW_URL = 'https://voteview.com/static/data/out/members/HS116_members.csv'

EXCLUDED_PUBLIC  = {'hubbywk1','racdif1','racdif2','racdif3','racdif4',
                    'workwhts','wlthwhts','intlwhts'}
EXCLUDED_PRIVATE = {'reborn','marwht','helpful','helpfulnv','helpfulv'}

# ── HuggingFace authentication ────────────────────────────────────────────────
# Required for gated models (Llama-3.3) and to download models not yet cached.
# Load from secrets.py if available, otherwise from environment variable
import os
try:
    from secrets import HF_TOKEN
except ImportError:
    HF_TOKEN = os.environ.get('HF_TOKEN', '')

# ── Models to run ─────────────────────────────────────────────────────────────
# force_quantize:
#   True  → always use 4-bit NF4 (required for 70-72B on 96 GB VRAM)
#   False → always use BF16 / FP16 (fine for ≤ 48B on 96 GB)
#   None  → auto-detect: NF4 when VRAM < 60 GB, else BF16
#
# Batch-size guidance (H100-96GB):
#   2B  BF16 → 500    24B BF16 → 90    32-34B BF16 → 64
#   70-72B NF4 → 32   (NF4 weights ~36-38 GB; 32 seqs × 128 toks ≈ 3-4 GB KV)
MODELS = [
    {
        # 72B BF16 = 144 GB — must quantize to NF4 (~36 GB)
        'local_path':     '/project/jevans/maxzhuyt/models/Qwen2.5-72B-Instruct',
        'hf_id':          'Qwen/Qwen2.5-72B-Instruct',
        'batch_size':     32,
        'label':          'Qwen2.5-72B',
        'force_quantize': True,
    },
    {
        # 24B BF16 = ~48 GB — fits natively, ~48 GB headroom
        'local_path':     '/project/jevans/maxzhuyt/models/Dolphin-Mistral-24B-Venice-Edition',
        'hf_id':          'dphn/Dolphin-Mistral-24B-Venice-Edition',
        'batch_size':     90,
        'label':          'Dolphin-24B-Venice',
        'force_quantize': False,
    },
    {
        # 2B BF16 = ~4 GB — huge headroom, push batch size up
        'local_path':     '/project/jevans/maxzhuyt/models/dolphin-2.9.4-gemma2-2b',
        'hf_id':          'dphn/dolphin-2.9.4-gemma2-2b',
        'batch_size':     500,
        'label':          'Dolphin-Gemma2-2B',
        'force_quantize': False,
    },
    {
        # 32B BF16 = 64 GB — fits with ~32 GB headroom
        'local_path':     '/project/jevans/maxzhuyt/models/Qwen3-32B',
        'hf_id':          'Qwen/Qwen3-32B',
        'batch_size':     64,
        'label':          'Qwen3-32B',
        'force_quantize': False,
    },
    {
        # 70B BF16 = 140 GB — must quantize to NF4 (~35 GB)
        'local_path':     '/project/jevans/maxzhuyt/models/Llama-3.3-70B-Instruct',
        'hf_id':          'meta-llama/Llama-3.3-70B-Instruct',
        'batch_size':     32,
        'label':          'Llama-3.3-70B',
        'force_quantize': True,
    },
    {
        # 34B BF16 = ~68 GB — fits on H100-96 with ~28 GB headroom
        'local_path':     '/project/jevans/maxzhuyt/models/dolphin-2.9.1-yi-1.5-34b',
        'hf_id':          'dphn/dolphin-2.9.1-yi-1.5-34b',
        'batch_size':     64,
        'label':          'Dolphin-Yi-34B',
        'force_quantize': False,
    },
]

# ── Google Drive output directory ─────────────────────────────────────────────
GDRIVE_DIR = '/content/drive/MyDrive/Polarization'


In [ ]:
# ── Public topics (126 GSS public-issue variables) ────────────────────────────
# Each entry: variable -> NaturalLanguageClause

PUBLIC_TOPICS = {
    'abdefect': 'whether a pregnant woman should be able to obtain a legal abortion if there is a strong chance of serious defect in the baby',
    'abhlth':   'whether a pregnant woman should be able to obtain a legal abortion if her health is seriously endangered',
    'abnomore': 'whether a married woman should be able to obtain a legal abortion if she does not want more children',
    'abpoor':   'whether a pregnant woman should be able to obtain a legal abortion if the family cannot afford more children',
    'abrape':   'whether a pregnant woman should be able to obtain a legal abortion if she became pregnant as a result of rape',
    'absingle': 'whether an unmarried woman should be able to obtain a legal abortion',
    'abhelp1':  'whether to help a family member or friend with arrangements for an abortion',
    'abhelp2':  'whether to help pay for an abortion',
    'abhelp3':  'whether to help pay for costs related to an abortion other than the procedure itself',
    'abhelp4':  'whether to provide emotional support to someone having an abortion',
    'advfront': 'whether scientific research that advances knowledge should be supported by the federal government',
    'balpos':   'the degree to which benefits of scientific research outweigh harmful results',
    'scientbe': 'whether scientists want to make life better for the average person',
    'scientgo': 'whether scientists work for the good of humanity',
    'scienthe': 'whether scientists help solve challenging problems',
    'affrmact': 'whether Black people should be given preference in hiring and promotion',
    'discaff':  'the likelihood that a white person will be passed over for a job while a less qualified Black person gets it',
    'discaffm': 'the likelihood that a man will be passed over for a job while a less qualified woman gets it',
    'discaffw': 'the likelihood that a woman will be passed over for a job while a less qualified man gets it',
    'fehire':   'whether employers should make special efforts to hire and promote women',
    'fejobaff': 'whether women should be given preference in hiring and promotion',
    'wrkwayup': 'whether Black people should work their way up without special favors',
    'carsgen':  'the degree of danger car pollution poses to the environment',
    'grncon':   'how concerned to be about environmental issues',
    'grnecon':  'whether we worry too much about the environment and not enough about prices and jobs',
    'grneffme': 'whether environmental problems have a direct effect on everyday life',
    'grnexagg': 'whether claims about environmental threats are exaggerated',
    'grnprice': 'how willing to pay higher prices to protect the environment',
    'grnprog':  'whether people worry too much about progress harming the environment',
    'grnsol':   'how willing to accept cuts in living standards to protect the environment',
    'grntaxes': 'how willing to pay higher taxes to protect the environment',
    'grwtharm': 'whether economic growth always harms the environment',
    'grwthelp': 'whether America needs economic growth to protect the environment',
    'harmsgrn': 'whether almost everything we do in modern life harms the environment',
    'helpharm': 'whether it is hard to know if the way you live helps or harms the environment',
    'ihlpgrn':  'whether to do what is right for the environment even when it costs more money or time',
    'impgrn':   'whether there are more important things in life than protecting the environment',
    'nobuygrn': 'how often to avoid buying certain products for environmental reasons',
    'othssame': 'whether there is no point in helping the environment unless others do the same',
    'scigrn':   'whether modern science will solve environmental problems with little change to our way of life',
    'tempgen1': 'the degree of danger climate change poses to the environment',
    'toodifme': 'whether it is too difficult to do much about the environment',
    'cappun':   'whether to favor or oppose the death penalty for murder',
    'courts':   'whether courts deal too harshly or not harshly enough with criminals',
    'polabuse': 'whether a policeman would be justified in striking a citizen who said vulgar things',
    'polattak': 'whether a policeman would be justified in striking a citizen who was attacking with fists',
    'polescap': 'whether a policeman would be justified in striking a citizen attempting to escape custody',
    'polhitok': 'whether there are situations in which to approve of police striking a citizen',
    'polmurdr': 'whether a policeman would be justified in striking a citizen being questioned as a murder suspect',
    'colath':   'whether an anti-religious person should be allowed to teach in college',
    'colcom':   'whether a communist teacher should be fired',
    'colhomo':  'whether a homosexual person should be allowed to teach in college',
    'colmil':   'whether a person advocating military rule should be allowed to teach in college',
    'colmslm':  'whether a Muslim clergyman who preaches hatred of the United States should be allowed to teach in college',
    'colrac':   'whether a person who believes Black people are genetically inferior should be allowed to teach in college',
    'conbus':   'the level of confidence in major companies',
    'conclerg': 'the level of confidence in organized religion',
    'coneduc':  'the level of confidence in education',
    'confed':   'the level of confidence in the executive branch of the federal government',
    'confinan': 'the level of confidence in banks and financial institutions',
    'conjudge': 'the level of confidence in the United States Supreme Court',
    'conlegis': 'the level of confidence in Congress',
    'dangroth': 'whether people with mental health problems should be forced to be hospitalized if dangerous to others',
    'dangrslf': 'whether people with mental health problems should be forced to be hospitalized if dangerous to themselves',
    'mustdoc':  'whether people with mental health problems should be forced by law to be examined by a doctor',
    'musthosp': 'whether people with mental health problems should be forced by law to be hospitalized',
    'mustmed':  'whether people with mental health problems should be forced by law to take medication',
    'viggrp':   'how willing to have a group home for people with mental health problems in the neighborhood',
    'divlaw':   'whether divorce should be easier or more difficult to obtain',
    'eqwlth':   'whether the government should reduce income differences between the rich and poor',
    'goveqinc': 'whether it is the government\'s responsibility to reduce income differences',
    'goveqinc1':'whether it is the government\'s responsibility to fix income differences',
    'govineq1': 'whether most politicians care about reducing income differences',
    'govineq2': 'how successful the government is at reducing income differences',
    'govchrst': 'whether the federal government should advocate Christian values',
    'govfnaid': 'whether the government should give financial aid to college students from low-income families',
    'govfnanc': 'whether the government should finance projects to create new jobs',
    'govunemp': 'whether the government should provide unemployment benefits',
    'helpblk':  'whether the government has a special obligation to help improve the living standards of Black people',
    'helpnot':  'whether the government should do more or less to solve the country\'s problems',
    'helppoor': 'whether the government should improve the standard of living of poor Americans',
    'helpsick': 'whether it is the government\'s responsibility to help people pay for medical care',
    'hlthgov':  'whether the government should provide only limited health care services',
    'grass':    'whether marijuana should be made legal',
    'gunlaw':   'whether to favor or oppose requiring a permit to buy a gun',
    'inteduc':  'how interested to be in local school issues',
    'ldctax':   'whether people in wealthy countries should pay extra taxes to help people in poor countries',
    'letdie1':  'whether doctors should be allowed by law to end an incurable patient\'s life if requested',
    'libath':   'whether an anti-religious book should be removed from the public library',
    'libcom':   'whether a communist\'s book should be removed from the public library',
    'libhomo':  'whether a book in favor of homosexuality should be removed from the public library',
    'libmil':   'whether a book advocating military rule should be removed from the public library',
    'libmslm':  'whether a book preaching hatred of the United States should be removed from the public library',
    'librac':   'whether a book saying Black people are inferior should be removed from the public library',
    'nataid':   'how much money to spend on foreign aid',
    'natarms':  'how much money to spend on the military armaments and defense',
    'natchld':  'how much money to spend on assistance for childcare',
    'natcrime': 'how much money to spend on halting the rising crime rate',
    'natdrug':  'how much money to spend on dealing with drug addiction',
    'nateduc':  'how much money to spend on improving the nation\'s education system',
    'natenvir': 'how much money to spend on improving and protecting the environment',
    'natfare':  'how much money to spend on welfare',
    'natheal':  'how much money to spend on improving and protecting the nation\'s health',
    'natmass':  'how much money to spend on mass transportation',
    'natpark':  'how much money to spend on parks and recreation',
    'natrace':  'how much money to spend on improving the conditions of Black people',
    'natroad':  'how much money to spend on highways and bridges',
    'natsci':   'how much money to spend on supporting scientific research',
    'natsoc':   'how much money to spend on social security',
    'natspac':  'how much money to spend on the space exploration program',
    'nextgen':  'whether science and technology will give more opportunities to the next generation',
    'oprace':   'how important being the right race is for getting ahead in life',
    'opsex':    'how important being born a man or woman is for getting ahead in life',
    'polviews': 'where to place oneself on the liberal to conservative scale',
    'pillok':   'whether birth control should be available to teenagers without parental approval',
    'sexeduc':  'whether to favor or oppose sex education in public schools',
    'prayer':   'whether to approve or disapprove of the Supreme Court ruling against Bible prayer in public schools',
    'religinf': 'whether the United States would be better if religion had less influence',
    'racopen':  'whether to vote for a law allowing homeowners to refuse to sell based on race',
    'spkath':   'whether an anti-religious person should be allowed to make a speech',
    'spkcom':   'whether a communist should be allowed to make a speech',
    'spkhomo':  'whether a homosexual person should be allowed to make a speech',
    'spkmil':   'whether a person advocating military rule should be allowed to make a speech',
    'spkmslm':  'whether a Muslim clergyman who preaches hatred of the United States should be allowed to make a speech',
    'spkrac':   'whether a person who believes Black people are inferior should be allowed to make a speech',
    'tax':      'whether the amount of federal income tax paid is too high or too low',
}

# GSS survey polarization: |mean_rep - mean_dem| / scale_range
GSS_PUBLIC_POLARIZATION = {
    'abhelp2': 0.472126, 'abnomore': 0.470704, 'abpoor':   0.446558,
    'absingle':0.443313, 'abhelp3': 0.439301, 'natrace':  0.418868,
    'helpblk': 0.415418, 'eqwlth':  0.413940, 'goveqinc1':0.393433,
    'cappun':  0.383457, 'grnexagg':0.376733, 'goveqinc': 0.372621,
    'polviews':0.359472, 'wrkwayup':0.358084, 'natenvir': 0.342124,
    'natarms': 0.332797, 'abhelp1': 0.330796, 'grncon':   0.329571,
    'tempgen1':0.329135, 'helpnot': 0.324098, 'govunemp': 0.318617,
    'grntaxes':0.315828, 'gunlaw':  0.315706, 'natfare':  0.313762,
    'religinf':0.312773, 'affrmact':0.308292, 'helppoor': 0.301537,
    'helpsick':0.293103, 'govchrst':0.291363, 'grnecon':  0.290854,
    'grnsol':  0.278682, 'grnprice':0.264658, 'govfnaid': 0.257589,
    'fejobaff':0.257080, 'grnprog': 0.256448, 'abdefect': 0.254661,
    'ldctax':  0.251146, 'govfnanc':0.250253, 'discaff':  0.232681,
    'confed':  0.228529, 'nataid':  0.224791, 'pillok':   0.220994,
    'grass':   0.214338, 'natchld': 0.211986, 'abrape':   0.202543,
    'natheal': 0.202046, 'fehire':  0.198659, 'polhitok': 0.197277,
    'hlthgov': 0.195918, 'natmass': 0.192238, 'polescap': 0.191410,
    'discaffm':0.189689, 'natcrime':0.176149, 'prayer':   0.175516,
    'carsgen': 0.175338, 'sexeduc': 0.174573, 'conjudge': 0.172035,
    'oprace':  0.169987, 'advfront':0.169797, 'nateduc':  0.169764,
    'colath':  0.162904, 'divlaw':  0.157948, 'impgrn':   0.157468,
    'racopen': 0.155145, 'abhelp4': 0.153885, 'conclerg': 0.151794,
    'grneffme':0.144125, 'letdie1': 0.143420, 'colcom':   0.138382,
    'natsci':  0.137750, 'natdrug': 0.132074, 'coneduc':  0.130912,
    'nobuygrn':0.128391, 'spkrac':  0.128370, 'harmsgrn': 0.126821,
    'scientgo':0.115940, 'abhlth':  0.113160, 'tax':      0.113159,
    'colrac':  0.108033, 'scienthe':0.107772, 'polattak': 0.107656,
    'discaffw':0.106765, 'grwthelp':0.103641, 'conlegis': 0.102199,
    'libath':  0.098789, 'libmslm': 0.095955, 'viggrp':   0.095450,
    'libcom':  0.094536, 'othssame':0.091075, 'opsex':    0.083842,
    'toodifme':0.083470, 'scientbe':0.083112, 'ihlpgrn':  0.078554,
    'spkcom':  0.078514, 'natpark': 0.078151, 'govineq2': 0.076053,
    'mustmed': 0.074419, 'colmslm': 0.070746, 'balpos':   0.068826,
    'courts':  0.068734, 'librac':  0.067736, 'grwtharm': 0.065489,
    'spkmslm': 0.065304, 'spkmil':  0.064760, 'nextgen':  0.060754,
    'govineq1':0.059366, 'libmil':  0.059326, 'scigrn':   0.057289,
    'spkath':  0.054078, 'musthosp':0.053507, 'conbus':   0.052471,
    'libhomo': 0.052173, 'polabuse':0.051213, 'colhomo':  0.049940,
    'polmurdr':0.049591, 'confinan':0.045844, 'inteduc':  0.042224,
    'colmil':  0.041305, 'natsoc':  0.036100, 'spkhomo':  0.031996,
    'helpharm':0.028944, 'dangrslf':0.026999, 'natroad':  0.014717,
    'natspac': 0.014073, 'mustdoc': 0.012724, 'dangroth': 0.006099,
}

print(f'Public topics: {len(PUBLIC_TOPICS)}, with polarization data: {len(GSS_PUBLIC_POLARIZATION)}')

In [ ]:
# ── Private life topics ───────────────────────────────────────────────────────
# Only variables with both a NaturalLanguageClause and GSS polarization data.
# Excludes: reborn, marwht, helpful, helpfulnv, helpfulv.

PRIVATE_TOPICS = {
    # Sexual behaviour
    'homosex':   'whether homosexual sex relations are wrong',
    'premarsx':  'whether sex before marriage is wrong',
    'teensex':   'whether sex before marriage for teenagers is wrong',
    'xmarsex':   'whether extramarital sex is wrong',
    'sexfreq':   'how often to have sex',
    'partners':  'how many sex partners to have in a year',
    'partnrs5':  'how many sex partners to have in the last five years',
    'frndsex':   'whether to have sex with a friend',
    'pikupsex':  'whether to have sex with a casual date',
    'paidsex':   'whether to have sex for pay',
    'othersex':  'whether to have sex with some other type of partner',
    'evpaidsx':  'whether to have ever had sex with someone who was paid or who paid for sex',
    'evstray':   'whether to have sex with someone other than a spouse while married',
    'matesex':   'whether a sex partner was a spouse or regular partner',
    'nummen':    'how many male sex partners to have since age 18',
    'numwomen':  'how many female sex partners to have since age 18',
    'acqntsex':  'whether to have sex with an acquaintance',
    'condom':    'whether to use a condom when having sex',
    # Religion
    'homosex':   'whether homosexual sex relations are wrong',  # already above
    'religimp':  'how important religion is in life',
    'relidimp':  'how important religion is to the respondent personally',
    'god':       'confidence in the existence of God',
    'savesoul':  'whether to try to convince others to accept Jesus',
    'pray':      'how often to pray',
    'attend':    'how often to attend religious services',
    'postlife':  'whether to believe in life after death',
    'bible':     'feelings about the Bible',
    'relpersn':  'to what extent to self-identify as a religious person',
    'sprtprsn':  'to what extent to self-identify as a spiritual person',
    'relexp':    'whether to have had a religious experience that changed life',
    'relidesc':  'how well religion describes one as a person',
    'relidins':  'to what extent to feel insulted when someone criticizes religion',
    'relidwe':   'how often to say we instead of they when talking about religion',
    'spanking':  'whether it is sometimes necessary to discipline a child with spanking',
    'popespks':  'whether the Pope is infallible on matters of faith and morals',
    # Family & lifestyle
    'marmakid':  'whether a single mother can raise a child as well as a married couple',
    'marpakid':  'whether a single father can raise a child as well as a married couple',
    'meovrwrk':  'whether family life suffers when men focus too much on work',
    'eldfnce':   'whether grandparents should help grandchildren financially',
    'marasian':  'how to feel about a close relative marrying an Asian American person',
    'richwork':  'whether to continue working if rich enough to live comfortably',
    'hunt':      'whether to go hunting',
    'recycle':   'how often to make a special effort to recycle',
    'raclive':   'whether there are people of the opposite race living in the neighborhood',
    'rfamlook':  'how many hours per week to spend looking after family members',
    'spfalook':  'how many hours per week a spouse spends looking after family members',
    'socbar':    'how often to go to a bar',
    'socfrend':  'how often to spend a social evening with friends',
    'socommun':  'how often to spend a social evening with neighbors',
    'socrel':    'how often to spend a social evening with relatives',
    'xmovie':    'whether to see an X-rated movie in the last year',
    'hrsrelax':  'how many hours per day to have to relax',
    'life':      'whether life is exciting, routine, or dull',
    'goodlife':  'whether standard of living will improve in the future',
    'kidssol':   "whether children's standard of living will be better or worse",
    'parsol':    "whether standard of living is better or worse than parents'",
    # Success factors
    'opwlth':    'how important coming from a wealthy family is for getting ahead in life',
    'oppared':   'how important having well-educated parents is for getting ahead in life',
    'opeduc':    'how important having a good education is for getting ahead in life',
    'ophrdwrk':  'how important hard work is for getting ahead in life',
    'opknow':    'how important knowing the right people is for getting ahead in life',
    'opclout':   'how important having political connections is for getting ahead in life',
    'oprelig':   'how important being the right religion is for getting ahead in life',
    # Science & technology
    'toofast':   'whether science makes our way of life change too fast',
    'intsci':    'how interested to be in new scientific discoveries',
    'inttech':   'how interested to be in new inventions and technologies',
    # Mental health
    'fammhneg':  'to what extent family holds negative attitudes about people with mental health problems',
    'othmhneg':  'to what extent acquaintances hold negative attitudes about people with mental health problems',
    # Misc
    'fear':      'whether to be afraid to walk alone at night in the neighborhood',
    'evcrack':   'whether to have ever used crack cocaine',
    'emailhr':   'how many hours per week to spend on email',
    'emailmin':  'how many minutes per week to spend on email',
    'wwwhr':     'how many hours per week to use the web',
    'wwwmin':    'how many minutes per week to use the web',
}
# Remove duplicate key if present (homosex duplicated above by accident)
PRIVATE_TOPICS = dict(PRIVATE_TOPICS)  # dedup via dict construction

GSS_PRIVATE_POLARIZATION = {
    'homosex':  0.324925, 'religimp': 0.227485, 'savesoul': 0.223262,
    'premarsx': 0.195853, 'relidimp': 0.193470, 'spanking': 0.192885,
    'god':      0.188705, 'teensex':  0.186717, 'relpersn': 0.175501,
    'pray':     0.166374, 'hunt':     0.146266, 'relidins': 0.141994,
    'xmarsex':  0.140302, 'marpakid': 0.136908, 'marmakid': 0.136141,
    'opwlth':   0.131215, 'relexp':   0.123355, 'frndsex':  0.117130,
    'attend':   0.114853, 'intsci':   0.108747, 'fear':     0.107324,
    'bible':    0.106812, 'relidwe':  0.105123, 'postlife': 0.102363,
    'oppared':  0.093081, 'condom':   0.086555, 'toofast':  0.083522,
    'inttech':  0.083510, 'xmovie':   0.081701, 'opclout':  0.080519,
    'evstray':  0.079299, 'sprtprsn': 0.071668, 'relidesc': 0.065814,
    'raclive':  0.065111, 'recycle':  0.059614, 'marasian': 0.057793,
    'ophrdwrk': 0.056994, 'opknow':   0.056276, 'partnrs5': 0.046878,
    'sexfreq':  0.036498, 'popespks': 0.035039, 'othersex': 0.032673,
    'pikupsex': 0.032525, 'goodlife': 0.032363, 'socbar':   0.029653,
    'socrel':   0.029416, 'opeduc':   0.027343, 'othmhneg': 0.025442,
    'meovrwrk': 0.023536, 'partners': 0.023363, 'emailmin': 0.020972,
    'paidsex':  0.019517, 'wwwhr':    0.018299, 'spfalook': 0.017463,
    'kidssol':  0.016835, 'wwwmin':   0.016791, 'fammhneg': 0.016722,
    'rfamlook': 0.013632, 'parsol':   0.013610, 'matesex':  0.013288,
    'evpaidsx': 0.012843, 'richwork': 0.012370, 'nummen':   0.012256,
    'socommun': 0.009651, 'eldfnce':  0.009503, 'evcrack':  0.008284,
    'hrsrelax': 0.007918, 'socfrend': 0.007049, 'life':     0.007037,
    'numwomen': 0.006707, 'oprelig':  0.004322, 'acqntsex': 0.002540,
    'emailhr':  0.001761,
}

print(f'Private topics: {len(PRIVATE_TOPICS)}, with polarization data: {len(GSS_PRIVATE_POLARIZATION)}')

In [ ]:
# ── Short display labels (≤12 chars) for visualization ───────────────────────
# Used instead of raw variable names in the scatter plot.

SHORT_LABELS = {
    # ── Public: abortion ──────────────────────────────────────────────────────
    'abdefect': 'Abort defect', 'abhlth':   'Abort health', 'abnomore': 'No more kids',
    'abpoor':   'Abort poor',   'abrape':   'Abort rape',   'absingle': 'Abort single',
    'abhelp1':  'Help w/abort', 'abhelp2':  'Pay 4 abort',  'abhelp3':  'Abort costs',
    'abhelp4':  'Abort emosup',
    # ── Public: science ───────────────────────────────────────────────────────
    'advfront': 'Fund sci',     'balpos':   'Sci benefits', 'scientbe': 'Sci for avg',
    'scientgo': 'Sci humanity', 'scienthe': 'Sci solves',   'nextgen':  'Sci+opportn',
    # ── Public: affirmative action ────────────────────────────────────────────
    'affrmact': 'Affirm Black', 'discaff':  'White passed', 'discaffm': 'Male passed',
    'discaffw': 'Fem passed',   'fehire':   'Hire women',   'fejobaff': 'Affirm women',
    'wrkwayup': 'No spec fav',
    # ── Public: environment ───────────────────────────────────────────────────
    'carsgen':  'Car pollutn',  'grncon':   'Env concern',  'grnecon':  'Env vs jobs',
    'grneffme': 'Env my life',  'grnexagg': 'Env alarmism', 'grnprice': 'Pay 4 env',
    'grnprog':  'Progress+env', 'grnsol':   'Std of livng', 'grntaxes': 'Tax 4 env',
    'grwtharm': 'Grwth=harm',   'grwthelp': 'Growth helps', 'harmsgrn': 'Life harms',
    'helpharm': 'Env unclear',  'ihlpgrn':  'I help env',   'impgrn':   'Other > env',
    'nobuygrn': 'Boycott env',  'othssame': 'Others same',  'scigrn':   'Sci env fix',
    'tempgen1': 'Climate risk', 'toodifme': 'Env too hard',
    # ── Public: crime & police ────────────────────────────────────────────────
    'cappun':   'Death penalt', 'courts':   'Harsh courts', 'polabuse': 'Police vulg',
    'polattak': 'Police attk',  'polescap': 'Police flee',  'polhitok': 'Police hit?',
    'polmurdr': 'Police murd',
    # ── Public: civil liberties ───────────────────────────────────────────────
    'colath':   'Teach athst',  'colcom':   'Teach commst', 'colhomo':  'Teach homo',
    'colmil':   'Teach militr', 'colmslm':  'Teach Muslim', 'colrac':   'Teach racist',
    'libath':   'Lib athst bk', 'libcom':   'Lib comm bk',  'libhomo':  'Lib homo bk',
    'libmil':   'Lib mil bk',   'libmslm':  'Lib Musl bk',  'librac':   'Lib race bk',
    'spkath':   'Speak athst',  'spkcom':   'Speak commst', 'spkhomo':  'Speak homo',
    'spkmil':   'Speak militr', 'spkmslm':  'Speak Muslim', 'spkrac':   'Speak racist',
    # ── Public: institutional confidence ─────────────────────────────────────
    'conbus':   'Confid corp',  'conclerg': 'Confid clrgy', 'coneduc':  'Confid educ',
    'confed':   'Confid feds',  'confinan': 'Confid banks', 'conjudge': 'Confid court',
    'conlegis': 'Confid Cong',
    # ── Public: mental health ─────────────────────────────────────────────────
    'dangroth': 'Hosp others',  'dangrslf': 'Hosp self',    'mustdoc':  'Force doctor',
    'musthosp': 'Force hosp',   'mustmed':  'Force meds',   'viggrp':   'Group home',
    # ── Public: economics & gov ───────────────────────────────────────────────
    'divlaw':   'Easy divorce', 'eqwlth':   'Income gap',   'goveqinc': 'Gov eq resp',
    'goveqinc1':'Gov eq duty',  'govineq1': 'Pols care eq', 'govineq2': 'Gov eq succ',
    'govchrst': 'Govt Christn', 'govfnaid': 'Aid students', 'govfnanc': 'Gov jobs',
    'govunemp': 'Unempl aid',   'helpblk':  'Help Black',   'helpnot':  'Gov do more',
    'helppoor': 'Help poor',    'helpsick': 'Med care gov', 'hlthgov':  'Limit health',
    'grass':    'Legal weed',   'gunlaw':   'Gun permit',   'inteduc':  'School intst',
    'ldctax':   'Natl aid tax', 'letdie1':  'Euthanasia',   'tax':      'Tax too high',
    # ── Public: spending (nat*) ───────────────────────────────────────────────
    'nataid':   'Spend F.aid',  'natarms':  'Spend defns',  'natchld':  'Spend child',
    'natcrime': 'Spend crime',  'natdrug':  'Spend drugs',  'nateduc':  'Spend educ',
    'natenvir': 'Spend envir',  'natfare':  'Spend welfre', 'natheal':  'Spend health',
    'natmass':  'Spend transt', 'natpark':  'Spend parks',  'natrace':  'Spend race',
    'natroad':  'Spend roads',  'natsci':   'Spend sci',    'natsoc':   'Spend SS',
    'natspac':  'Spend space',
    # ── Public: misc ─────────────────────────────────────────────────────────
    'oprace':   'Race impt',    'opsex':    'Gender impt',  'polviews': 'Lib-Con self',
    'pillok':   'Teen contcpt', 'sexeduc':  'Sex ed schl',  'prayer':   'School pray',
    'religinf': 'Less relig',   'racopen':  'Race housing',
    # ── Private: sexual behaviour ─────────────────────────────────────────────
    'homosex':  'Homosex OK',   'premarsx': 'Premarital',   'teensex':  'Teen sex OK',
    'xmarsex':  'Extramarital', 'sexfreq':  'Sex freq',     'partners': '# partners',
    'partnrs5': '5yr partners', 'frndsex':  'Sex w friend', 'pikupsex': 'Pickup sex',
    'paidsex':  'Pay for sex',  'othersex': 'Other sex',    'evpaidsx': 'Ever paid sx',
    'evstray':  'Cheating',     'matesex':  'Sex w spouse', 'nummen':   'Male partnrs',
    'numwomen': 'Fem partnrs',  'acqntsex': 'Sex acquaint', 'condom':   'Use condom',
    # ── Private: religion ─────────────────────────────────────────────────────
    'religimp': 'Relig import', 'relidimp': 'Relig imp v2', 'god':      'Belief God',
    'savesoul': 'Save souls',   'pray':     'Pray often',   'attend':   'Church freq',
    'postlife': 'Afterlife',    'bible':    'Bible view',   'relpersn': 'Relig person',
    'sprtprsn': 'Spiritual',    'relexp':   'Relig exprnc', 'relidesc': 'Relig descr',
    'relidins': 'Relig insult', 'relidwe':  "Relig 'we'",   'spanking': 'Spanking OK',
    'popespks': 'Pope infallb',
    # ── Private: family & lifestyle ───────────────────────────────────────────
    'marmakid': 'Single mom',   'marpakid': 'Single dad',   'meovrwrk': 'Men overwork',
    'eldfnce':  'Grndpr help',  'marasian': 'Marry Asian',  'richwork': 'Stop if rich',
    'hunt':     'Hunting',      'recycle':  'Recycling',    'raclive':  'Mixed neighb',
    'rfamlook': 'Fam care hrs', 'spfalook': 'Sp fam care',  'socbar':   'Bar nights',
    'socfrend': 'Eve friends',  'socommun': 'Eve neighbor', 'socrel':   'Eve family',
    'xmovie':   'X-rated film', 'hrsrelax': 'Relax hrs',    'life':     'Life excitng',
    'goodlife': 'Good life',    'kidssol':  'Kids life',     'parsol':   'Vs parents',
    # ── Private: success factors ──────────────────────────────────────────────
    'opwlth':   'Wealth impt',  'oppared':  'Educ parents', 'opeduc':   'Educ import',
    'ophrdwrk': 'Hard work',    'opknow':   'Right people', 'opclout':  'Polit clout',
    'oprelig':  'Relig ahead',
    # ── Private: science & tech ───────────────────────────────────────────────
    'toofast':  'Sci too fast', 'intsci':   'Sci intrest',  'inttech':  'Tech intrest',
    # ── Private: mental health ────────────────────────────────────────────────
    'fammhneg': 'Fam MH stig',  'othmhneg': 'Acquaint MH',
    # ── Private: misc ─────────────────────────────────────────────────────────
    'fear':     'Fear night',   'evcrack':  'Crack use',    'emailhr':  'Email hrs/wk',
    'emailmin': 'Email min/wk', 'wwwhr':    'Web hrs/wk',   'wwwmin':   'Web min/wk',
}

# Verify all labels are ≤12 chars
too_long = {k: v for k, v in SHORT_LABELS.items() if len(v) > 12}
if too_long:
    print('WARNING — labels exceeding 12 chars:', too_long)
else:
    print(f'SHORT_LABELS: {len(SHORT_LABELS)} entries, all ≤12 chars.')

In [ ]:
# ── Helper: model loading ─────────────────────────────────────────────────────
import os
os.environ['PYTORCH_ALLOC_CONF'] = 'expandable_segments:True'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

import gc
import torch
import numpy as np
import pandas as pd
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

# ── HuggingFace login ─────────────────────────────────────────────────────────
# Needed for gated models (Llama-3.3-70B) and to download models not yet cached.
import huggingface_hub
huggingface_hub.login(token=HF_TOKEN, add_to_git_credential=False)
print('HuggingFace login OK.')


def load_model_and_tokenizer(local_path, hf_model_id, force_quantize=None):
    """
    Load model from local_path; fall back to HuggingFace download.
    Auto-enables 4-bit NF4 when VRAM < 60 GB (covers A100-40GB which reports ~40.5 GB).
    Pass force_quantize=True to always use NF4 (required for 70-72B on 96 GB H100).
    """
    vram_gb = (torch.cuda.get_device_properties(0).total_memory / 1e9
               if torch.cuda.is_available() else 0)
    quantize = force_quantize if force_quantize is not None else (0 < vram_gb < 60)
    use_local = os.path.isdir(local_path)
    src = local_path if use_local else hf_model_id
    print(f'Loading: {src}  |  VRAM {vram_gb:.1f} GB  |  4-bit NF4: {quantize}')

    dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
    tok = AutoTokenizer.from_pretrained(src, use_fast=True, local_files_only=use_local)
    tok.padding_side = 'left'
    tok.truncation_side = 'left'
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
        tok.pad_token_id = tok.eos_token_id

    kwargs = dict(device_map='auto', local_files_only=use_local, attn_implementation='sdpa')
    if quantize:
        kwargs['quantization_config'] = BitsAndBytesConfig(
            load_in_4bit=True, bnb_4bit_quant_type='nf4',
            bnb_4bit_compute_dtype=dtype, bnb_4bit_use_double_quant=True)
    else:
        kwargs['dtype'] = dtype

    model = AutoModelForCausalLM.from_pretrained(src, **kwargs)
    model.generation_config.pad_token_id = tok.pad_token_id
    print(f'Ready — {model.config.num_hidden_layers} layers, '
          f'{model.config.num_attention_heads} heads')
    return model, tok


print('Model loading function defined.')


In [ ]:
# ── Helper: activation extraction ────────────────────────────────────────────
# Hooks on self_attn.o_proj input to capture per-head activations at last token.

@torch.no_grad()
def extract_heads_batched(model, tokenizer, texts, system_msg,
                          batch_size=32, max_length=128):
    """
    Returns np.ndarray of shape (N, L, H, D_head).
    Hook captures o_proj.input[0]: [B, Seq, H*D] → reshape → take last token.
    """
    model.eval()
    L = model.config.num_hidden_layers
    H = model.config.num_attention_heads
    D = getattr(model.config, 'head_dim', model.config.hidden_size // H)
    layer_out = [None] * L

    def make_hook(li):
        def hook(module, inp, out):
            x = inp[0].detach().view(inp[0].shape[0], inp[0].shape[1], H, D)
            layer_out[li] = x[:, -1, :, :].float().cpu().numpy()
        return hook

    hooks = [model.model.layers[li].self_attn.o_proj.register_forward_hook(make_hook(li))
             for li in range(L)]

    def fmt(text):
        if getattr(tokenizer, 'chat_template', None):
            try:
                return tokenizer.apply_chat_template(
                    [{'role': 'system', 'content': system_msg},
                     {'role': 'user',   'content': text}],
                    tokenize=False, add_generation_prompt=True)
            except Exception:
                try:
                    return tokenizer.apply_chat_template(
                        [{'role': 'user', 'content': f'{system_msg}\n\n{text}'}],
                        tokenize=False, add_generation_prompt=True)
                except Exception:
                    pass
        return f'{system_msg}\n\n{text}'

    all_acts = []
    try:
        for i in range(0, len(texts), batch_size):
            batch = [fmt(t) for t in texts[i:i+batch_size]]
            enc = tokenizer(batch, return_tensors='pt', padding=True,
                            truncation=True, max_length=max_length).to(model.device)
            model(**enc)
            all_acts.append(np.stack(layer_out, axis=1))
    finally:
        for h in hooks:
            h.remove()
    return np.concatenate(all_acts, axis=0)


print('Activation extraction function defined.')

In [ ]:
# ── Helper: metric computation ────────────────────────────────────────────────
import warnings
from scipy.linalg import inv, LinAlgError
from scipy.spatial.distance import mahalanobis
from sklearn.decomposition import PCA
from sklearn.metrics import davies_bouldin_score
from joblib import Parallel, delayed


def _head_metrics(head_data, labels, gv=(100, 200), pca_dim=15, eigen_dim=10, pca_save=30):
    """
    All metrics for one (layer, head).

    Returns
    -------
    metrics : dict
        Scalar summary metrics.
    pca_scores : np.ndarray, shape (N, pca_save), float32
        First pca_save PCA components for all N politicians.
        Fit on D+R subset; since we only load D/R, this covers all N rows.
    """
    valid = np.isin(labels, gv)
    X, y = head_data[valid], labels[valid]
    n, D = X.shape

    # ── Eigenvalue metrics on raw space ──────────────────────────────────────
    ne = min(eigen_dim, D, n - 1)
    try:
        with warnings.catch_warnings():
            warnings.filterwarnings('ignore')
            pca_e = PCA(n_components=ne).fit(X - X.mean(0))
        ev = pca_e.explained_variance_
        disp      = float(ev.sum())
        intrinsic = float(ev.sum()**2 / (ev**2).sum()) if (ev**2).sum() > 0 else 0.0
        pc1       = float(pca_e.explained_variance_ratio_[0])
    except Exception:
        disp = intrinsic = pc1 = 0.0

    # ── Davies-Bouldin ────────────────────────────────────────────────────────
    try:
        db = (float(davies_bouldin_score(X, y))
              if (y == gv[0]).sum() >= 2 and (y == gv[1]).sum() >= 2
              else float('nan'))
    except Exception:
        db = float('nan')

    # ── PCA-Mahalanobis + save first pca_save components ─────────────────────
    nm_save = min(pca_save, D, n - 1)
    nm      = min(pca_dim, nm_save)
    pca_scores = np.zeros((n, nm_save), dtype=np.float32)
    mahal = 0.0
    try:
        with warnings.catch_warnings():
            warnings.filterwarnings('ignore')
            pca = PCA(n_components=nm_save).fit(X)
        Xp = pca.transform(X).astype(np.float32)   # (N, nm_save)
        pca_scores = Xp
        g1, g2 = Xp[y == gv[0], :nm], Xp[y == gv[1], :nm]
        if len(g1) > 5 and len(g2) > 5:
            c1  = np.cov(g1.T, ddof=1) if len(g1) > 1 else np.eye(nm)
            c2  = np.cov(g2.T, ddof=1) if len(g2) > 1 else np.eye(nm)
            cov = (c1 + c2) / 2 + np.eye(nm) * 1e-6
            mahal = float(mahalanobis(g1.mean(0), g2.mean(0), inv(cov)))
    except Exception:
        mahal = 0.0

    return dict(mahal=mahal, disp=disp, intrinsic=intrinsic, pc1=pc1, db=db), pca_scores


def compute_topic_metrics(acts, labels, pca_dim=15, eigen_dim=10, pca_save=30, n_jobs=-1):
    """
    Average all metrics across all (L, H) heads.

    Returns
    -------
    avg_metrics : dict
        Scalar averages (same keys as before).
    head_mahal : np.ndarray, shape (L, H), float32
        Per-head Mahalanobis distance grid.
    pca_grid : np.ndarray, shape (L, H, N, pca_save), float16
        Per-head PCA projections. Saved as float16 to halve storage.
        Use float32 cast before metric recomputation.
    """
    N, L, H, D = acts.shape
    heads = [(l, h, acts[:, l, h, :]) for l in range(L) for h in range(H)]

    res = Parallel(n_jobs=n_jobs)(
        delayed(_head_metrics)(
            hd, labels, pca_dim=pca_dim, eigen_dim=eigen_dim, pca_save=pca_save)
        for _, _, hd in heads)

    metrics_list = [r[0] for r in res]
    pca_list     = [r[1] for r in res]   # each (N, K)

    dbs = [m['db'] for m in metrics_list if not np.isnan(m['db'])]
    avg_metrics = {
        'Avg_Mahal':            float(np.mean([m['mahal']     for m in metrics_list])),
        'Avg_Total_Dispersion': float(np.mean([m['disp']      for m in metrics_list])),
        'Avg_Intrinsic_Dim':    float(np.mean([m['intrinsic'] for m in metrics_list])),
        'Avg_PC1_Ratio':        float(np.mean([m['pc1']       for m in metrics_list])),
        'Avg_Davies_Bouldin':   float(np.mean(dbs)) if dbs else float('nan'),
    }

    # (L, H) per-head Mahalanobis grid
    head_mahal = np.array(
        [m['mahal'] for m in metrics_list], dtype=np.float32).reshape(L, H)

    # (L, H, N, K) PCA grid in float16
    K        = pca_list[0].shape[1] if pca_list else pca_save
    pca_grid = np.zeros((L, H, N, K), dtype=np.float16)
    for idx, (l, h, _) in enumerate(heads):
        pca_grid[l, h] = pca_list[idx].astype(np.float16)

    return avg_metrics, head_mahal, pca_grid


print('Metric functions defined.')

In [ ]:
# ── Load politicians ──────────────────────────────────────────────────────────
import io, urllib.request

def load_politicians(local_csv, voteview_url):
    if os.path.isfile(local_csv):
        df = pd.read_csv(local_csv)
        df = df[df['party_code'].isin([100, 200])].dropna(subset=['fullname'])
        names  = df['fullname'].tolist()
        labels = df['party_code'].values.astype(int)
    else:
        print('Downloading from VoteView...')
        with urllib.request.urlopen(voteview_url) as r:
            df = pd.read_csv(io.StringIO(r.read().decode('utf-8')))
        df = df[(df['congress']==116) & df['party_code'].isin([100,200])].dropna(subset=['bioname'])
        def fmt(b):
            p = [x.strip() for x in b.split(',')]
            return f'{p[1].title()} {p[0].title()}' if len(p)==2 else b.title()
        names  = [fmt(n) for n in df['bioname']]
        labels = df['party_code'].values.astype(int)
    d, r = int((labels==100).sum()), int((labels==200).sum())
    print(f'Politicians: {len(names)}  (D={d}, R={r})')
    return names, labels


politician_names, party_labels = load_politicians(LOCAL_POLITICIAN_CSV, VOTEVIEW_URL)

In [ ]:
# ── Mount Google Drive ────────────────────────────────────────────────────────
import os, datetime

try:
    from google.colab import drive
    drive.mount('/content/drive')
    os.makedirs(GDRIVE_DIR, exist_ok=True)
    print(f'Drive mounted.  Results → {GDRIVE_DIR}')
except ImportError:
    print('Not in Colab — results will be saved locally.')
    GDRIVE_DIR = '.'

In [ ]:
# ── Multi-model analysis loop ─────────────────────────────────────────────────
# Saving layout
# ─────────────────────────────────────────────────────────────────────────────
# GDRIVE_DIR/
#   {label}_{condition}_{ts}.csv                  ← complete metrics CSV per (model, condition)
#   results/
#     {label}_{condition}_{ts}/                   ← per-(model, condition) subfolder
#       {label}_{condition}_{topic}.npz           ← per-topic PCA scores (L×H×N×K)
#       {label}_{condition}_{ts}_headwise.npz     ← stacked (T,L,H) Mahal grid

import time, datetime


def make_prompt(name, nlc, condition):
    if condition == 'stance':
        return f"What is {name}'s position on {nlc}?"
    elif condition == 'rhetorical':
        return f'Generate a statement by {name} on {nlc}.'
    else:  # survey
        return f'If asked in a national survey about {nlc}, how would {name} respond?'


# Build combined topic list once (shared across all models and conditions)
all_items = []
if 'public' in CATEGORIES:
    for var, nlc in PUBLIC_TOPICS.items():
        if var not in EXCLUDED_PUBLIC and var in GSS_PUBLIC_POLARIZATION:
            all_items.append((var, nlc, GSS_PUBLIC_POLARIZATION[var], 'public'))
if 'private' in CATEGORIES:
    for var, nlc in PRIVATE_TOPICS.items():
        if var not in EXCLUDED_PRIVATE and var in GSS_PRIVATE_POLARIZATION:
            all_items.append((var, nlc, GSS_PRIVATE_POLARIZATION[var], 'private'))

n_pub = sum(1 for *_, c in all_items if c == 'public')
n_prv = sum(1 for *_, c in all_items if c == 'private')
print(f'Topics: {len(all_items)}  (public={n_pub}, private={n_prv})')
print(f'Models: {[m["label"] for m in MODELS]}')
print(f'Conditions: {CONDITIONS}\n')

# ── Outer loop: model ─────────────────────────────────────────────────────────
# Load each model once; iterate all conditions before unloading.
all_dfs = {}   # keyed by (label, condition)

for model_cfg in MODELS:
    label = model_cfg['label']
    bs    = model_cfg['batch_size']
    print(f'\n{"="*60}')
    print(f'  {label}  |  batch_size={bs}  |  conditions={CONDITIONS}')
    print(f'{"="*60}')

    model, tokenizer = load_model_and_tokenizer(
        model_cfg['local_path'], model_cfg['hf_id'],
        force_quantize=model_cfg.get('force_quantize'))

    # ── Inner loop: condition ─────────────────────────────────────────────────
    for condition in CONDITIONS:
        print(f'\n  ── Condition: {condition} ──')

        # Fresh timestamp per (model, condition) so files don't collide
        ts        = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')
        model_dir = os.path.join(GDRIVE_DIR, 'results', f'{label}_{condition}_{ts}')
        os.makedirs(model_dir, exist_ok=True)
        print(f'  Secondary outputs → {model_dir}')

        results         = []
        head_mahal_list = []
        t0 = time.time()

        for i, (var, nlc, gss_pol, cat) in enumerate(all_items):
            prompts = [make_prompt(n, nlc, condition) for n in politician_names]
            acts = extract_heads_batched(
                model, tokenizer, prompts, SYSTEM_MSG,
                batch_size=bs, max_length=MAX_LENGTH)

            avg_metrics, head_mahal, pca_grid = compute_topic_metrics(
                acts, party_labels,
                pca_dim=PCA_DIM, eigen_dim=EIGEN_PCA_DIM, pca_save=PCA_SAVE)

            # ── Save per-topic PCA scores ──────────────────────────────────────
            pca_path = os.path.join(model_dir, f'{label}_{condition}_{var}.npz')
            np.savez_compressed(
                pca_path,
                pca30        = pca_grid,      # (L, H, N, PCA_SAVE) float16
                party_labels = party_labels,  # (N,) int
                label        = label,
                topic        = var,
                condition    = condition,
            )

            # ── Collect scalar metrics ─────────────────────────────────────────
            m = avg_metrics.copy()
            m.update(topic=var, category=cat, gss_polarization=gss_pol,
                     model=label, condition=condition)
            results.append(m)
            head_mahal_list.append(head_mahal)   # (L, H) float32

            elapsed = time.time() - t0
            eta     = elapsed / (i + 1) * (len(all_items) - i - 1)
            print(f'  [{i+1:3d}/{len(all_items)}] {var:12s} [{cat[:3]}] '
                  f'mahal={m["Avg_Mahal"]:.3f}  gss={gss_pol:.3f}  ETA {eta/60:.1f}m')

            del acts, pca_grid
            gc.collect()
            torch.cuda.empty_cache()

        df_cond = pd.DataFrame(results)
        all_dfs[(label, condition)] = df_cond

        # ── Save complete metrics CSV ──────────────────────────────────────────
        csv_path = os.path.join(GDRIVE_DIR, f'{label}_{condition}_{ts}.csv')
        df_cond.to_csv(csv_path, index=False)
        print(f'\n  CSV saved:          {csv_path}')

        # ── Save combined headwise Mahalanobis grid ────────────────────────────
        head_mahal_stack = np.stack(head_mahal_list, axis=0)   # (T, L, H)
        hw_path = os.path.join(model_dir, f'{label}_{condition}_{ts}_headwise.npz')
        np.savez_compressed(
            hw_path,
            head_mahal   = head_mahal_stack,
            topics       = np.array([v for v, *_ in all_items]),
            categories   = np.array([c for *_, c in all_items]),
            party_labels = party_labels,
            label        = label,
            condition    = condition,
        )
        print(f'  Headwise grid:      {hw_path}')
        print(f'  PCA files ({len(all_items)} topics): {model_dir}/')

        del head_mahal_list
        gc.collect()
        torch.cuda.empty_cache()

    # ── Unload model after all conditions ─────────────────────────────────────
    del model, tokenizer
    gc.collect()
    torch.cuda.empty_cache()
    print(f'  Unloaded {label}')

# Combine for downstream cells
df_all = pd.concat(all_dfs.values(), ignore_index=True)
df = df_all
print(f'\nAll done — {len(df_all)} rows across {len(all_dfs)} (model, condition) combos.')
df_all.groupby(['model', 'condition'])[['Avg_Mahal', 'gss_polarization']].mean().round(3)


In [ ]:
# ── Per-(model, condition) correlation summary ────────────────────────────────
from scipy.stats import pearsonr, spearmanr

metrics_meta = [
    ('Avg_Mahal',            'PCA-Mahalanobis',    False),
    ('Avg_Total_Dispersion', 'Total Dispersion',   False),
    ('Avg_Intrinsic_Dim',    'Intrinsic Dim',      False),
    ('Avg_PC1_Ratio',        'PC1 Var Ratio',      False),
    ('Avg_Davies_Bouldin',   'Davies-Bouldin (↓)', True),
]

for mdl in df_all['model'].unique():
    for cond in CONDITIONS:
        df_mc = df_all[(df_all['model'] == mdl) & (df_all['condition'] == cond)]
        if df_mc.empty:
            continue
        print(f'\n{"="*60}\n  {mdl}  [{cond}]\n{"="*60}')
        for cat in ['public', 'private', 'all']:
            sub = df_mc if cat == 'all' else df_mc[df_mc['category'] == cat]
            sub = sub.dropna(subset=['gss_polarization'])
            print(f'\n  ── {cat.upper()} ({len(sub)} topics) ──')
            print(f'  {"Metric":22s}  {"Pearson r":>10s}  {"Spearman ρ":>10s}')
            print('  ' + '-' * 47)
            for col, lbl, negate in metrics_meta:
                s2 = sub.dropna(subset=[col])
                if len(s2) < 5:
                    continue
                vals = -s2[col] if negate else s2[col]
                r,   _ = pearsonr(vals,  s2['gss_polarization'])
                rho, _ = spearmanr(vals, s2['gss_polarization'])
                print(f'  {lbl:22s}  {r:10.3f}  {rho:10.3f}')


In [ ]:
# ── Plotly scatter: Mahalanobis vs Total Dispersion ───────────────────────────
# Dots: uniform steelblue. Labels: collision-avoidance repulsion, ≤12 chars.
# Quadrant lines at medians. Category in hover text.
# When df_all contains multiple conditions, averages across them per topic/model.

import plotly.graph_objects as go


def repel_labels(xs, ys, n_iter=250, repel=0.025, attract=0.07, seed=42):
    """Spring-based label repulsion in normalised [0,1] space."""
    xs, ys = np.asarray(xs, float), np.asarray(ys, float)
    xr = (xs.max() - xs.min()) or 1
    yr = (ys.max() - ys.min()) or 1
    xn, yn = (xs - xs.min()) / xr, (ys - ys.min()) / yr

    rng = np.random.default_rng(seed)
    lx = xn + rng.uniform(-0.01, 0.01, len(xn))
    ly = yn + rng.uniform(-0.01, 0.01, len(yn))
    thr = 0.035

    for _ in range(n_iter):
        for i in range(len(lx)):
            dx, dy = lx[i] - lx, ly[i] - ly
            dist = np.hypot(dx, dy) + 1e-9
            mask = (dist < thr); mask[i] = False
            if mask.any():
                lx[i] += (repel * dx[mask] / dist[mask]).sum()
                ly[i] += (repel * dy[mask] / dist[mask]).sum()
            lx[i] += attract * (xn[i] - lx[i])
            ly[i] += attract * (yn[i] - ly[i])

    return lx * xr + xs.min(), ly * yr + ys.min()


# Average across conditions per (model, topic) so one dot per topic
plot_df = (df_all
           .dropna(subset=['Avg_Mahal', 'Avg_Total_Dispersion', 'gss_polarization'])
           .groupby(['model', 'topic', 'category', 'gss_polarization'], as_index=False)
           [['Avg_Mahal', 'Avg_Total_Dispersion']].mean())

xs      = plot_df['Avg_Mahal'].values
ys      = plot_df['Avg_Total_Dispersion'].values
cats    = plot_df['category'].values
gss     = plot_df['gss_polarization'].values
vars_   = plot_df['topic'].values
lbls    = [SHORT_LABELS.get(v, v[:12]) for v in vars_]

print('Computing label positions...')
lx, ly = repel_labels(xs, ys)

med_x, med_y = np.median(xs), np.median(ys)

model_label_str = ', '.join(sorted(df_all['model'].unique()))
cond_str        = ' + '.join(CONDITIONS)

fig = go.Figure()

# Dots — uniform steelblue, circle for public, diamond for private
for cat, sym in [('public', 'circle'), ('private', 'diamond')]:
    mask = cats == cat
    fig.add_trace(go.Scatter(
        x=xs[mask], y=ys[mask],
        mode='markers',
        marker=dict(color='steelblue', size=8, symbol=sym,
                    opacity=0.85, line=dict(color='white', width=0.5)),
        text=[f'<b>{SHORT_LABELS.get(v, v)}</b> [{c}]<br>'
              f'Mahal={x:.3f}  Disp={y:.1f}  GSS={g:.3f}'
              for v, c, x, y, g in
              zip(vars_[mask], cats[mask], xs[mask], ys[mask], gss[mask])],
        hoverinfo='text', name=cat.capitalize(),
    ))

# Leader lines
for xi, yi, lxi, lyi in zip(xs, ys, lx, ly):
    if abs(xi-lxi) > 0.001 or abs(yi-lyi) > 0.5:
        fig.add_shape(type='line', x0=xi, y0=yi, x1=lxi, y1=lyi,
                      line=dict(color='rgba(120,120,120,0.25)', width=0.7))

# Text labels at repelled positions
fig.add_trace(go.Scatter(
    x=lx, y=ly, mode='text', text=lbls,
    textfont=dict(size=7, color='#333'),
    hoverinfo='skip', showlegend=False))

# Quadrant lines
xpad = (xs.max()-xs.min()) * 0.03
ypad = (ys.max()-ys.min()) * 0.03
xrng = [xs.min()-xpad, xs.max()+xpad]
yrng = [ys.min()-ypad, ys.max()+ypad]
for x0, y0, x1, y1 in [
    (med_x, yrng[0], med_x, yrng[1]),
    (xrng[0], med_y, xrng[1], med_y),
]:
    fig.add_shape(type='line', x0=x0, y0=y0, x1=x1, y1=y1,
                  line=dict(color='rgba(180,60,60,0.45)', width=1, dash='dot'))

fig.update_layout(
    title=dict(
        text=(f'Partisan Activation Structure per Topic (avg across conditions)<br>'
              f'<sup>{model_label_str} | conditions: {cond_str} | '
              f'PCA-{PCA_DIM} Mahalanobis vs Total Dispersion | '
              f'● public  ◆ private</sup>'),
        font=dict(size=13)),
    xaxis=dict(title='Avg PCA-Mahalanobis (D vs R)',
               showgrid=True, gridcolor='#eee', range=xrng),
    yaxis=dict(title='Avg Total Dispersion (Σ eigenvalues)',
               showgrid=True, gridcolor='#eee', range=yrng),
    plot_bgcolor='white',
    width=1300, height=900,
    legend=dict(x=0.01, y=0.99),
    margin=dict(l=60, r=40, t=90, b=60),
)

fig.show()
fig.write_html('polarization_scatter.html')
print('Saved → polarization_scatter.html')


In [ ]:
# ── Run summary ───────────────────────────────────────────────────────────────
# Per-(model, condition) CSVs are already saved inside the main loop (cell-11).
# This cell prints a final summary and descriptive stats.

import datetime
print(f'Run complete: {datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")}')
print(f'Models: {list(df_all["model"].unique())}')
print(f'Conditions: {list(df_all["condition"].unique())}')
print(f'Total rows: {len(df_all)}\n')

print('Files written:')
for mdl in df_all['model'].unique():
    for cond in df_all['condition'].unique():
        print(f'  {GDRIVE_DIR}/{mdl}_{cond}_<ts>.csv')
        print(f'  {GDRIVE_DIR}/results/{mdl}_{cond}_<ts>/   ← PCA arrays + headwise grid')

print()
print(df_all[['model', 'condition', 'topic', 'category', 'Avg_Mahal',
              'Avg_Total_Dispersion', 'Avg_Intrinsic_Dim', 'Avg_PC1_Ratio',
              'Avg_Davies_Bouldin', 'gss_polarization']].describe().round(3))
